# LIME: Local interpretable model-agnostic explanations

The LIME method is the second model-agnostic perturbation method we encounter that attempts to explain what drives model outputs.

It operates by first splitting an input image into properly well defined segments (called "super-pixels"). It then perturbs the input image by deactivating a few of those super-pixels and observing how the model prediction for the most important classes change.

This process is repeated many time, therby generating a data set of perturbations and resulting class-probabilities. This new dataset can be used to train a regular model (e.g. linear regression with regularization) to predict the model output based on what super-pixels are turned on or off. That new model has essentially learned which super-pixels are important!

You may need to install the python library `lime` for this notebook to work:

In [ ]:
!pip install lime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.applications import resnet50
from tensorflow.keras.preprocessing import image

import keras_hub
from PIL import Image as PILImage

from lime import lime_image
from skimage.segmentation import mark_boundaries


## Load pretrained model and data

In [ ]:
# We're loading the resnet50 model, as it was trained on IMAGENET data
model = resnet50.ResNet50(weights='imagenet')

# This auxiliary function ensures our image is of desired size and
# tensor shape
def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    # We don't preprocess yet because LIME needs the raw image
    # to perturb it; we preprocess inside the prediction function.
    return x, img


In [ ]:
# We're using the image loading procedure as in the other 05_* notebook:
import gdown
file_id = "1YbfxA5xlRYpifmMvrrJexdkNMa6_xx6z"
url = f'https://drive.google.com/uc?id={file_id}'

output = 'xception_image_data.npz' # Rename it to whatever extension you need
gdown.download(url, output, quiet=False)

example_images = np.load("xception_image_data.npz")
X = example_images["X"]
y = example_images["y"]

X.shape

In [ ]:
# Let's pick image 19, a dog in front of a dishwasher
img_array = np.expand_dims(X[19,:,:,:], axis=0)
PILImage.fromarray(img_array[0])

In [ ]:
print(keras_hub.utils.decode_imagenet_predictions(batch_predict(img_array)))
#PILImage.fromarray(img[0])

## Setting up the LIME explanation

In [ ]:
# LIME needs a function that takes a numpy array and returns probabilities
# We create this function using this wrapper
def batch_predict(images):
    # Apply ResNet50 specific preprocessing (scaling/mean subtraction)
    preprocessed_images = resnet50.preprocess_input(images.copy())
    return model.predict(preprocessed_images)

# Initialize LIME Image Explainer
explainer = lime_image.LimeImageExplainer()

# This function does the heavy lifting:
# It finds appropriate super-pixels, performs `num_samples` perturbations around
# the original input image and each time computes how the model prediction changes
# (through the wrapper `batch_predict`).
# The resulting object contains all information necessary to produce importance
# images
explanation = explainer.explain_instance(img_array[0].astype('double'),
                                         batch_predict,
                                         num_features = 20,
                                         top_labels=5,
                                         hide_color=0,
                                         num_samples=800)

In [ ]:
# Visualize the positive segments (Pros)
temp, mask = explanation.get_image_and_mask(explanation.top_labels[0],
                                            positive_only=False,
                                            num_features=5,
                                            hide_rest=False)

plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title("LIME Explanation for Top Class")
plt.show()

In [ ]:
# In this section, the default definition of superpixels (semantic segments) is
# changed. Fewer, larger super-pixels should be used (e.g., ~20-50 segments)
# This code is experimental and has not yet been thoroughly tested.
from skimage.segmentation import quickshift
custom_segmenter = lambda img: quickshift(img, kernel_size=8, max_dist=200, ratio=0.2)

# Initialize LIME Image Explainer
explainer2 = lime_image.LimeImageExplainer()

# This function does the heavy lifting:
# It finds appropriate super-pixels, performs `num_samples` perturbations around
# the original input image and each time computes how the model prediction changes
# (through the wrapper `batch_predict`).
# The resulting object contains all information necessary to produce importance
# images
explanation = explainer2.explain_instance(img_array[0].astype('double'),
                                         batch_predict,
                                         num_features = 20,
                                         # we provide a custom image segmenter here
                                         # this defines the superpixels
                                         segmentation_fn=custom_segmenter,
                                         top_labels=5,
                                         hide_color=0,
                                         num_samples=800)